# 05 — Selección del modelo juez (§5.7)

**Autor:** Jason Sepúlveda S.
**Proyecto:** Evaluación experimental de la extracción automática y la representación
persistente de decisiones organizacionales frente a Recuperación Aumentada (RAG)
**Depende de:** [`04-jss-anotacion-f1-calibracion.ipynb`](04-jss-anotacion-f1-calibracion.ipynb)
(ES2015 anotada y adjudicada) · [propuesta §5.7](../docs/propuesta.md) ·
[metodología](../docs/avances/06-jss-metodologia.md) · issue #82

---

> **Estado: decisión tomada. El juez es `gpt-oss:20b`.** Las etiquetas humanas están
> completas (issue #82), los tres candidatos se corrieron sobre ellas y la regla de decisión
> se aplicó sin cambiar los umbrales fijados de antemano (§9). El modelo elegido queda fijado
> en `[judge].model` de `config/experiments.toml`.

## Qué hace este notebook

1. Explica por qué el Experimento B necesita un juez automático y qué se exige de él (§1–§2).
2. Presenta los candidatos, los modelos descartados y las razones (§3).
3. Muestra los umbrales **fijados antes de mirar ningún resultado**, leídos de la
   configuración y no reescritos aquí (§4).
4. Justifica por qué el listón de acuerdo del juez (kappa 0,70) es de otra naturaleza que el
   de la anotación (§5), por qué la calibración usa solo ES2015 (§6) y por qué la corrección
   de las citas no se le pide al juez (§7).
5. Carga el conjunto de calibración y, si está completo, corre los candidatos y aplica la
   regla de decisión (§8–§9).

## Qué NO hace este notebook

- **No calcula.** Toda cifra viene de una función con prueba de `src/afg/`
  ([`CONTRIBUTING.md` §7.1](../CONTRIBUTING.md)); el notebook importa, llama y explica.
- **No etiqueta ni inventa etiquetas.** Las etiquetas de calibración son siempre humanas.
- **No toca series de evaluación ni series de desarrollo con anotaciones pendientes** (§6).


In [1]:
from dataclasses import asdict

import pandas as pd

from afg.evaluation import citation
from afg.evaluation.judge_calibration import load_thresholds
from afg.evaluation.judge_calibration_agreement import AGREEMENT_PATH, load_agreement
from afg.evaluation.judge_calibration_generation import CALIBRATION_PATH
from afg.evaluation.judge_calibration_results import RESULTS_PATH, load_calibration_results
from afg.evaluation.judge_calibration_runner import load_candidates
from afg.evaluation.judge_calibration_set import (
    TEMPLATE_PATH,
    calibration_series,
    development_series,
    load_calibration_items,
    summarize_calibration_set,
)
from afg.evaluation.judge_calibration_worksheet import WORKSHEET_PATH
from afg.shared.config import load_experiments_config
from afg.shared.paths import PROJECT_ROOT

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 140)

# Guarda: el notebook solo necesita la configuracion versionada y la plantilla del conjunto
# de calibracion; no lee transcripciones ni el corpus. Falla con un mensaje claro si falta.
if not TEMPLATE_PATH.exists():
    raise SystemExit(
        f"No se encontro la plantilla del conjunto de calibracion en {TEMPLATE_PATH}.\n"
        "El clon esta incompleto: restaura data/processed/judge-calibration/ con git."
    )

CONFIG = load_experiments_config()
THRESHOLDS = load_thresholds()
print(f"Configuracion en  {(PROJECT_ROOT / 'config/experiments.toml').relative_to(PROJECT_ROOT)}")
print(f"Plantilla en      {TEMPLATE_PATH.relative_to(PROJECT_ROOT)}")

Configuracion en  config/experiments.toml
Plantilla en      data/processed/judge-calibration/_plantilla-calibracion-juez.csv


---

# 1. Por qué hace falta un juez

El Experimento B compara las tres condiciones (C1, C2, C3) sobre un banco de preguntas
estratificado (§5.5). Cada pregunta se responde con cada condición y cada configuración se
corre varias veces (§5.8), de modo que el experimento produce del orden de **900 respuestas**:
100 preguntas × 3 condiciones × 3 corridas. Calificar cada una a mano, contra su respuesta de
referencia y su evidencia, es inviable para un equipo de tres personas; por eso §5.7 prevé un
**juez automático** y, a cambio, exige validar con personas una fracción de lo que juzga.

Los dos parámetros que fijan ese compromiso salen de la configuración:

In [2]:
shared = CONFIG["shared_budget"]
evaluation = CONFIG["evaluation"]
pd.DataFrame(
    [
        ("Corridas por configuración (§5.8)", shared["runs_per_config"]),
        ("Fracción mínima con validación humana (§5.7)", evaluation["human_validation_fraction"]),
        ("Modelo generativo / extractor", shared["generative_model"]),
    ],
    columns=["parámetro", "valor"],
)

,parámetro,valor
0,Corridas por configuración (§5.8),3
1,Fracción mínima con validación humana (§5.7),0.2
2,Modelo generativo / extractor,qwen3:8b


Un juez automático solo vale si se comprueba que califica como lo haría una persona. §5.7 lo
formula como regla: *si la concordancia juez-humano es baja, el juicio automático se
descarta y se reporta únicamente la evaluación humana sobre la submuestra*. Este notebook es
la comprobación previa: elige el modelo con evidencia propia y deja explícito el caso en que
ninguno merece el cargo.

---

# 2. Criterios de selección y literatura

Ningún benchmark público mide a los candidatos en esta tarea (respuesta contra referencia, con
abstenciones y citas). Lo que sí existe es literatura sobre **cómo fallan los jueces**, y cada
criterio de selección responde a un fallo documentado:

| Fallo documentado | Fuente | Criterio que lo comprueba |
|---|---|---|
| Un modelo reconoce y favorece sus propias generaciones (sesgo de autopreferencia) | Panickssery et al. (2024), arXiv:2404.13076 | Juez de una familia ajena a toda etapa de OpenKOS (§3) |
| Sesgos de posición y de verbosidad; un juez fuerte puede acordar con personas en más del 80 % de los casos | Zheng et al. (2023), arXiv:2306.05685 | Consistencia al invertir el orden; sensibilidad al largo |
| El orden en que se presentan las respuestas altera el veredicto | Wang et al. (2024), arXiv:2305.17926 | Consistencia al invertir el orden referencia/respuesta |
| El acuerdo porcentual oculta diferencias; conviene medir con kappa de Cohen, y los modelos pequeños son menos fiables | Thakur et al. (2024), arXiv:2406.12624 | Kappa de Cohen con intervalo bootstrap, no solo exactitud |
| El desempeño del juez varía entre tareas y ningún modelo es el mejor en todas | Bavaresco et al. (2024), arXiv:2406.18403 | Calibración propia sobre la tarea propia, no reputación |
| Los jueces fallan en pares donde la corrección es objetiva y difícil de distinguir | Tan et al. (2024), arXiv:2410.12784 | Exactitud en el estrato sin respuesta (E4) y estabilidad entre semillas |
| Existen jueces pequeños especializados en evaluación | Atla (2025), arXiv:2501.17195 | Se incluye un candidato especializado (§3) |

Tres criterios no vienen de la literatura sino del uso: la **salida debe ser utilizable** (JSON
válido), la **estabilidad entre semillas** (el mismo prompt con otra semilla no debe cambiar el
veredicto) y la **latencia**, que desempata entre candidatos equivalentes.

---

# 3. Candidatos y modelos descartados

La lista se verificó en la biblioteca de Ollama el 2026-10-14. Todos se ejecutan en local
(§5.9); el mismo prompt, la temperatura 0, la semilla fija y el esquema JSON valen para todos.

## 3.1 Candidatos

In [3]:
roles = ["principal", "especializado en evaluación (base Llama 3.1 8B)", "reserva"]
pd.DataFrame({"modelo": load_candidates(), "papel": roles})

,modelo,papel
0,mistral-small3.2:24b,principal
1,atla/selene-mini:q8_0,especializado en evaluación (base Llama 3.1 8B)
2,gpt-oss:20b,reserva


## 3.2 Descartados y por qué

| Modelo | Razón |
|---|---|
| `phi4:14b` | Contexto de 16 K: no caben la pregunta, el pasaje de transcripción, la referencia y la respuesta con holgura para respuestas largas |
| Llama 3.3 70B | 43 GB: no cabe en el hardware disponible |
| Destilaciones de DeepSeek-R1 de base Qwen | Heredan la familia Qwen, la del extractor |
| Familia **Qwen** | Es el extractor de OpenKOS (`qwen3:8b`); un juez de la misma familia reabre el sesgo de autopreferencia |
| Familia **Gemma** | Es el adjudicador interno de OpenKOS; mismo riesgo, un paso más abajo en la cadena |

La regla de §5.7 es que el juez sea de una *familia distinta* de la del extractor.
`require_distinct_model_families` solo rechaza el caso trivial (el mismo identificador); la
exclusión de familias completas es una decisión de este protocolo y queda registrada aquí.

---

# 4. Umbrales prerregistrados

Los umbrales se fijaron **antes de correr ningún candidato** y no se ajustan después de ver
resultados. Viven en `config/experiments.toml` (`[judge.calibration]`) y se leen con
`load_thresholds()`; la tabla siguiente es esa lectura, no una copia escrita a mano.

In [4]:
meaning = {
    "min_kappa_correct": "kappa de Cohen contra las etiquetas humanas, `correct` (mínimo)",
    "min_position_consistency": "mismo veredicto al invertir el orden (mínimo)",
    "min_seed_consistency": "mismo veredicto en todas las semillas (mínimo)",
    "min_e4_accuracy": "exactitud en el estrato sin respuesta E4 (mínimo)",
    "min_json_valid_rate": "tasa de salidas JSON válidas (mínimo)",
    "max_verbosity_flip_rate": "veredictos que cambian al duplicar una respuesta mala (máximo)",
    "seeds": "semillas de la prueba de estabilidad",
}
pd.DataFrame(
    [(name, value, meaning[name]) for name, value in asdict(THRESHOLDS).items()],
    columns=["umbral", "valor", "qué mide"],
)

,umbral,valor,qué mide
0,min_kappa_correct,0.7,"kappa de Cohen contra las etiquetas humanas, `correct` (mínimo)"
1,min_position_consistency,0.9,mismo veredicto al invertir el orden (mínimo)
2,min_seed_consistency,0.95,mismo veredicto en todas las semillas (mínimo)
3,min_e4_accuracy,0.8,exactitud en el estrato sin respuesta E4 (mínimo)
4,min_json_valid_rate,0.99,tasa de salidas JSON válidas (mínimo)
5,max_verbosity_flip_rate,0.1,veredictos que cambian al duplicar una respuesta mala (máximo)
6,seeds,"(42, 43, 44)",semillas de la prueba de estabilidad


**Regla de decisión.** Entre los candidatos que cumplen *todos* los umbrales gana el de mayor
kappa en `correct`; si los intervalos de kappa se solapan, gana el de menor latencia. Si
ninguno cumple, **el juez automático se descarta** y §5.7 se cumple solo con la submuestra
humana. La regla está implementada en `decide_judge` y se aplica en la sección 9.

---

# 5. Por qué el listón de kappa del juez es 0,70

En la Fase 1 de la anotación, el kappa de estado sobre ES2015 fue de **0,094**
([notebook 04](04-jss-anotacion-f1-calibracion.ipynb)). Un listón de 0,70 para el juez podría
parecer incoherente con ese valor. No lo es, porque miden cosas de naturaleza distinta:

- **La anotación pregunta si una frase es una decisión.** Es un constructo subjetivo: depende
  de cómo se lea el contexto de la reunión, y por eso la guía necesitó corregirse. Un kappa
  bajo allí diagnosticaba una guía incompleta.
- **El juez compara una respuesta con una referencia ya escrita y comprueba sus afirmaciones
  contra un pasaje de la transcripción.** La pregunta es si dos textos dicen lo mismo sobre un
  hecho concreto y si el pasaje lo respalda. Es una tarea casi objetiva: dos personas
  atentas coinciden con facilidad, y por eso se puede exigir mucho a quien la automatice.

El listón de 0,70 es el mismo que usa la verificación humana del 20 % en la
[metodología](../docs/avances/06-jss-metodologia.md): lo que se le pide al juez no es más
indulgente que lo que se le pide a una persona que revisa.

---

# 6. Por qué la calibración usa solo ES2015

La calibración exige que personas lean preguntas, referencias y fragmentos de transcripción
de una serie. Eso solo es admisible sobre series de **desarrollo**
([ADR 0005](../docs/decisions/0005-adr-development-evaluation-split.md)), pero no todas las de
desarrollo están libres:

- **ES2015** ya está anotada y adjudicada; leerla no compromete ninguna anotación pendiente.
- **IS1004** se anota de forma independiente en la Fase 3 (#30); exponer su contenido antes
  rompería esa independencia.
- **TS3009** tiene la recalibración sin terminar, por la misma razón.
- Las series de control y el banco de evaluación (#32, #33) no se tocan.

La restricción no es una convención: la configuración la declara y el cargador la hace
cumplir. Si la configuración nombrara una serie fuera del desarrollo, la lectura falla.

In [5]:
pd.DataFrame(
    [
        ("Series de desarrollo (corpus.toml)", ", ".join(sorted(development_series()))),
        ("Series de la calibración (experiments.toml)", ", ".join(sorted(calibration_series()))),
    ],
    columns=["conjunto", "series"],
)

,conjunto,series
0,Series de desarrollo (corpus.toml),"ES2015, IS1004, TS3009"
1,Series de la calibración (experiments.toml),ES2015


---

# 7. La corrección de las citas no se juzga

§5.6 define la cita correcta como "la fuente citada respalda lo afirmado". El juez lee un pasaje de la
transcripción alrededor de la evidencia de *referencia*, pero **no sabe qué fuentes citó la
respuesta**, así que no puede verificarlo; pedírselo lo haría adivinar. La
corrección de las citas se decide con una función determinista que compara los actos de diálogo
citados con la evidencia de referencia, y `judge_answer` devuelve siempre `citation_correct =
None`. Su criterio, tal como lo documenta el código:

In [6]:
print(citation.citation_correct.__doc__)

Whether every citation in ``answer`` is backed by the question's reference evidence.

Returns ``None`` when the rate does not apply: an abstention, an E4 (unanswerable)
question, or a question without reference evidence. An answer to an answerable
question with no citation at all is ``False``: section 6.2 counts an unsourced claim
as a failure even when the claim is correct.



---

# 8. El conjunto de calibración

## 8.1 Cómo se construye

1. **Preguntas.** Unas 20 preguntas sobre ES2015, escritas por una persona del equipo a partir
   de las decisiones adjudicadas, que ya traen respuesta y evidencia de referencia, en los
   cuatro estratos. Se escriben en una hoja prellenada (`afg judge calibration-init`) donde la
   evidencia (`reference_evidence`) ya viene resuelta; la guía está en
   `docs/anotacion/calibracion-juez.md`.
2. **Respuestas.** Dos por pregunta, generadas con el modelo generativo (`qwen3:8b`): una
   *con fundamento* (el contexto es el pasaje que cubre la evidencia de referencia) y otra
   *sin fundamento* (un pasaje de ES2015 que no contiene esa evidencia, elegido de forma
   determinista desde la semilla). En la respuesta sin fundamento se le exige al modelo que
   responda sin abstenerse: si se le permite, se abstiene casi siempre (23 de 42 respuestas en
   una primera generación), y el conjunto solo mediría si el juez rechaza abstenciones. Así se
   produce la falla que §5.6 considera más peligrosa: una respuesta segura que la evidencia no
   respalda. En E4, que no tiene evidencia, la segunda respuesta se genera sin contexto y
   ambas pueden abstenerse, porque ahí abstenerse es lo correcto. La variante que produjo cada respuesta queda en un archivo aparte
   (`calibracion-juez.variantes.csv`) para comprobar que haya respuestas correctas e
   incorrectas en proporción similar; no se le muestra al juez, y el orden de las filas se baraja con
   la semilla para que tampoco lo delate. A quien etiqueta sí se le reveló por otra vía: el
   sufijo del identificador de cada ítem (ver la limitación en la sección 9.2).
3. **Etiquetas.** Cerca de 40 ítems, etiquetados por una persona (`correct` y
   `evidence_supported`), con el mismo método que el juez (ver abajo); una segunda persona etiqueta 15 de forma independiente para comprobar
   que la etiqueta humana es confiable. **Las etiquetas son siempre humanas.**

## 8.1.1 Qué lee el juez, y qué lee quien etiqueta

Las personas que anotan leen la evidencia enlazada y, cuando no basta, la transcripción
congelada. El juez sigue el mismo método. Recibe, en este orden: la pregunta, un **pasaje de la
transcripción**, y la respuesta de referencia y la del sistema (que se intercambian de lugar en
la prueba de orden; el pasaje no se mueve). El pasaje son los turnos de habla alrededor de los
actos de diálogo de la evidencia de referencia, con una ventana de `evidence_window_turns`
turnos a cada lado (sección `[judge]` de la configuración; la misma para todos los candidatos,
fijada antes de ver resultados). Las ventanas que se solapan se unen y las reuniones se
concatenan en orden cronológico.

- `correct`: la respuesta **coincide con la de referencia**.
- `evidence_supported`: **cada afirmación de la respuesta está respaldada por el pasaje**.
- En E4 no hay pasaje: el material no contiene la respuesta, y las dos etiquetas son verdaderas
  solo si la respuesta se abstiene.

El archivo que etiqueta la persona trae ese mismo pasaje (`pasaje_transcripcion`) y la ruta de
la transcripción completa (`archivos_transcripcion`) por si el pasaje no alcanza.

## 8.2 Estado actual

In [7]:
questions_file = (
    WORKSHEET_PATH.relative_to(PROJECT_ROOT) if WORKSHEET_PATH.exists() else "(aún no existe)"
)
calibration_file = (
    CALIBRATION_PATH.relative_to(PROJECT_ROOT) if CALIBRATION_PATH.exists() else "(aún no existe)"
)
print(f"Hoja de preguntas:          {questions_file}")
print(f"Conjunto de calibración:   {calibration_file}")

items = load_calibration_items(CALIBRATION_PATH) if CALIBRATION_PATH.exists() else []
summary = summarize_calibration_set(items)
print(f"Ítems: {summary.n_items} · con etiquetas humanas: {summary.n_labeled}")
if summary.n_items:
    display(
        pd.DataFrame(
            {
                "variante de respuesta": summary.by_variant,
            }
        ).T
    )
    display(pd.DataFrame({"estrato": summary.by_stratum}).T)

Hoja de preguntas:          data/processed/judge-calibration/ES2015.preguntas.csv
Conjunto de calibración:   data/processed/judge-calibration/calibracion-juez.csv
Ítems: 42 · con etiquetas humanas: 42


,grounded,no_context,ungrounded
variante de respuesta,17,4,21


,E1_point_fact,E2_current_state,E3_evolution,E4_unanswerable
estrato,28,4,2,8


---

# 9. Resultados

Para cada candidato se juzga cada ítem etiquetado en **ambos órdenes de presentación** y con
**cada semilla**, y se añade la **prueba de largo**: las respuestas que las personas
etiquetaron como incorrectas se alargan al doble, de forma determinista, y se vuelven a
juzgar. Se mide la latencia de cada llamada y se cuentan las salidas inutilizables. Esa
corrida es lenta y la hace `uv run afg judge calibration-run`, que exige etiquetas humanas en
todos los ítems y guarda los resultados en `reports/judge-calibration/resultados.json`. Este
notebook no llama a ningún modelo: solo lee ese archivo.

In [8]:
stored = None
if not RESULTS_PATH.exists():
    print(
        "pendiente: aún no hay resultados guardados (#82).\n"
        "Se generan con `uv run afg judge calibration-run`, que exige que todos los ítems "
        "tengan etiquetas humanas; este notebook solo los lee."
    )
else:
    stored = load_calibration_results(RESULTS_PATH)
    print(f"Resultados sobre {stored.n_items} ítems etiquetados ({RESULTS_PATH.name}).")
    table = pd.DataFrame(
        [
            {
                "modelo": run.model,
                "kappa": run.kappa,
                "IC inf": run.kappa_lower,
                "IC sup": run.kappa_upper,
                "posición": run.position_consistency,
                "semillas": run.seed_consistency,
                "E4": run.e4_accuracy,
                "JSON válido": run.json_valid_rate,
                "sesgo de largo": run.verbosity_flip_rate,
                "kappa evidencia*": run.evidence_kappa,
                "latencia (s)": run.latency_seconds,
                "llamadas": run.calls,
                "fallos": run.failures,
            }
            for run in stored.candidates
        ]
    )
    display(table.round(3))
    print("* kappa de `evidence_supported`: informativo, sin umbral prerregistrado.")

Resultados sobre 42 ítems etiquetados (resultados.json).


,modelo,kappa,IC inf,IC sup,posición,semillas,E4,JSON válido,sesgo de largo,kappa evidencia*,latencia (s),llamadas,fallos
0,mistral-small3.2:24b,0.951,0.843,1.000,0.952,1.0,1.0,1.0,0.056,0.234,5.499,270,0
1,atla/selene-mini:q8_0,0.485,0.222,0.738,0.738,1.0,1.0,1.0,0.000,0.043,3.004,270,0
2,gpt-oss:20b,0.951,0.843,1.000,0.952,1.0,1.0,1.0,0.000,0.806,3.224,270,0


* kappa de `evidence_supported`: informativo, sin umbral prerregistrado.


## 9.1 Decisión

In [9]:
if stored is None:
    print("pendiente: faltan los resultados guardados (#82). Nada que registrar todavía.")
else:
    print(f"Resultado: {stored.outcome}")
    print(f"Juez elegido: {stored.selected}")
    for model, failed in stored.failures.items():
        print(f"  {model}: " + (", ".join(failed) if failed else "cumple todos los umbrales"))

Resultado: selected
Juez elegido: gpt-oss:20b
  mistral-small3.2:24b: cumple todos los umbrales
  atla/selene-mini:q8_0: kappa_correct, position_consistency
  gpt-oss:20b: cumple todos los umbrales


**Lectura del resultado.**

- **`atla/selene-mini:q8_0` no cumple.** Su kappa en `correct` (0,49) y su consistencia al
  invertir el orden (0,74) quedan bajo los umbrales: cambia de veredicto según dónde aparezca
  la respuesta, que es el sesgo de posición descrito en la sección 2. Ser un modelo
  especializado en evaluación no bastó en esta tarea.
- **`mistral-small3.2:24b` y `gpt-oss:20b` cumplen todos los umbrales** con el mismo kappa
  (0,95) y el mismo intervalo. Como los intervalos se solapan, la regla desempata por
  latencia: `gpt-oss:20b` responde en 3,2 s por llamada frente a 5,5 s. En unas 900
  respuestas juzgadas en ambos órdenes, la diferencia es de poco más de una hora de cómputo
  (1.800 llamadas × 2,3 s).
- **El kappa de `evidence_supported` apunta en la misma dirección**, aunque no decide nada
  porque no tiene umbral prerregistrado: `gpt-oss:20b` alcanza 0,81, `mistral-small3.2:24b`
  0,23 y `atla/selene-mini:q8_0` 0,04. El candidato elegido por latencia es también el único
  que distingue de forma fiable una respuesta respaldada por el pasaje de una que no lo está.
  Se informa como dato, no como criterio añadido después de ver los resultados.

`gpt-oss:20b` figuraba como reserva en la sección 3; la calibración propia, y no la
reputación previa de cada modelo, es lo que decidió.

## 9.2 Acuerdo entre las dos personas que etiquetan

Que el juez coincida con las etiquetas humanas solo tiene sentido si esas etiquetas son
confiables. Por eso una segunda persona (`gm`) etiquetó por separado una muestra de 15 ítems
del conjunto (ids ordenados, barajados con la semilla 42), sin ver las etiquetas de la
primera (`jss`). El umbral de kappa es el mismo de la sección 4 y se lee de la configuración,
no se repite aquí. El resultado lo calcula `uv run afg judge calibration-agreement`; este
notebook solo lo lee.

In [10]:
if not AGREEMENT_PATH.exists():
    print(
        "pendiente: aún no hay acuerdo humano guardado.\n"
        "Se genera con `uv run afg judge calibration-agreement`."
    )
else:
    columns, min_kappa, labelers = load_agreement(AGREEMENT_PATH)
    print(f"Acuerdo {labelers[0]} vs {labelers[1]} (kappa mínimo {min_kappa:.2f})")
    display(
        pd.DataFrame(
            [
                {
                    "columna": a.column,
                    "ítems": a.n_items,
                    "coinciden": a.n_agree,
                    "kappa": a.kappa,
                    "cumple": a.passes,
                    "discrepancias": ", ".join(a.disagreements) or "-",
                }
                for a in columns
            ]
        ).round(3)
    )

Acuerdo jss vs gm (kappa mínimo 0.70)


,columna,ítems,coinciden,kappa,cumple,discrepancias
0,correct,15,15,1.000,True,-
1,evidence_supported,15,14,0.865,True,ES2015.q008.grounded


### Limitación: el `id` revela la variante de la respuesta

Los ids de los ítems terminan en `.grounded` o `.ungrounded`, de modo que mostraron a ambas
personas (`jss` y `gm`) cómo se generó cada respuesta, a pesar de que la variante se guarda
aparte en un archivo auxiliar precisamente para no revelarla. El equipo decidió declararlo en
lugar de reetiquetar.

Hay una evidencia que atenúa el problema: `gm` etiquetó `ES2015.q008.ungrounded` como correcta
y respaldada, y `ES2015.q008.grounded` como no respaldada por la evidencia. Las etiquetas
siguieron el contenido de la respuesta y no el sufijo del id. Aun así, queda un riesgo
residual: tanto el acuerdo entre personas como el acuerdo juez-humano pueden estar
ligeramente inflados, y no se puede cuantificar cuánto.

---

# 10. Límites y siguiente paso

- **Muestra pequeña.** Unos 40 ítems dan intervalos de kappa anchos. Por eso la regla de
  decisión usa el intervalo para desempatar y por eso el umbral se fijó sobre el punto
  estimado antes de ver datos; no se aflojó ni se reinterpretó.
- **Una sola serie.** ES2015 tiene pocas relaciones de evolución, así que E3 pesa poco en la
  calibración. Lo que se mide es si el juez compara bien una respuesta con su referencia, no
  la dificultad de cada estrato.
- **Respuestas de un solo generador.** Las respuestas de calibración las produce `qwen3:8b`;
  las del Experimento B vendrán de las tres condiciones. Es una aproximación declarada.
- **El identificador revela la variante.** Ver la sección 9.2: se declara como amenaza a la
  validez y no se reetiquetó.
- **Siguiente paso.** El Experimento B usa `gpt-oss:20b` como juez (fijado en la
  configuración) y conserva la verificación humana del 20 % que exige §5.7: si en esa
  submuestra la concordancia juez-humano cae bajo 0,70, el juicio automático se descarta.
  Queda por registrar la decisión de la sección 9.1 en la
  [metodología](../docs/avances/06-jss-metodologia.md).